# Rajasthan Royals SupeRR Selector 2.0: Comprehensive Exploratory Data Analysis & Match State Reconstruction
## Objective
Identify and rank:
1. **Top 10 Most Reliable Batters Under Pressure**
2. **Top 10 Most Reliable Bowlers Under Pressure**

This initial notebook covers:
- Inspection of all competition CSV files and entity relationships.
- Missing values and data quality checks.
- Ball-by-ball match situation reconstruction (scores, wickets in hand, target, RRR, match phases).
- Statistical analysis of scoring rates, extras, dismissals, and high-pressure chase dynamics.
- Clean, reproducible workflow using modular `src/` packages.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Project root setup
PROJECT_ROOT = Path('.').resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_raw_datasets, get_dataset_summary
from src.match_state import reconstruct_match_situations

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'Segoe UI', 'DejaVu Sans', 'Arial'
%matplotlib inline


## 1. Dataset Overview & Schema Integrity
We load all 6 competition datasets without mutating original raw records:
- `competitions.csv`: Match formats and seasons (2022-2026).
- `teams.csv`: International and franchise team mappings.
- `matches.csv`: Match metadata, venues, toss, knockouts, DLS revised targets.
- `playing_eleven.csv`: Lineups, batting positions, captaincy, and impact players.
- `competition_team_player_mapping.csv`: Canonical player identifiers and names.
- `ball_by_ball.csv`: 200,999 delivery-level records.


In [ ]:
datasets = load_raw_datasets()
summary_df = get_dataset_summary(datasets)
summary_df


## 2. Match Coverage & Integrity Findings
- **883 matches** in `matches.csv` vs **867 matches** in `ball_by_ball.csv`.
- The 16 matches lacking ball-by-ball records are all verified to be **Match Abandoned** without a ball bowled (e.g., rain washouts).
- **23 matches** contain Duckworth-Lewis-Stern (DLS) revised targets (`revised_target`).
- **Super Overs**: 38 deliveries occurred in Innings 3 and 4, representing peak pressure scenarios.


In [ ]:
matches_comp = datasets['matches'].merge(datasets['competitions'], on='comp_id', how='left')
matches_comp['tournament_type'] = np.where(
    matches_comp['comp_name'].str.contains('IPL', case=False, na=False),
    'IPL',
    'T20I International'
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
season_type_counts = matches_comp.groupby(['season', 'tournament_type']).size().unstack(fill_value=0)
season_type_counts.plot(kind='bar', stacked=True, ax=axes[0], color=['#FF4081', '#1976D2'], edgecolor='black', alpha=0.85)
axes[0].set_title('Matches by Season & Tournament Type', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Matches')
axes[0].grid(axis='y', linestyle='--', alpha=0.7)

stage_counts = matches_comp['stage'].value_counts()
axes[1].pie(stage_counts, labels=stage_counts.index, autopct='%1.1f%%', colors=['#42A5F5', '#66BB6A', '#FFA726', '#AB47BC'], startangle=140)
axes[1].set_title('Match Stage Distribution', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## 3. Delivery Mechanics: Runs, Extras & Wicket Attribution
Crucial cricket domain rules identified:
1. **Total runs per ball**: `runs` + `extras1_runs` + `extras_2_runs`.
2. **Legal balls**: Deliveries where `extras_type` is `wide` or `Overstep` are illegal balls and must be re-bowled.
3. **Bowler-credited wickets**: `Caught`, `Bowled`, `LBW`, `Stump`, and `Hit Wicket`. Run-outs and retired-outs are excluded.
4. **Batter balls faced**: Wides are not counted; legal balls and no-balls are faced.


In [ ]:
bbb = datasets['ball_by_ball']
print("Runs distribution off the bat:")
print(bbb['runs'].value_counts(normalize=True).round(3) * 100)

print("\nExtras type frequency:")
print(bbb['extras_type'].value_counts(dropna=False))

print("\nDismissal types:")
print(bbb['dismissal_type'].value_counts(dropna=False))


## 4. Ball-by-Ball Match Situation Reconstruction
Using `src/match_state.py`, we reconstruct:
- Pre-delivery score (`score_before`) and wickets fallen (`wickets_before`).
- Wickets in hand (`wickets_in_hand_before = 10 - wickets_before`).
- Legal deliveries remaining (`legal_balls_remaining_before`).
- Target score and Required Run Rate (`rrr_before`) for chase innings.
- Match phase segmentation: Powerplay (1-6), Middle (7-15), Death (16-20).


In [ ]:
reconstructed_df = reconstruct_match_situations(datasets['ball_by_ball'], datasets['matches'])
reconstructed_df[[
    'match_id', 'innings', 'overid', 'over_ball_num', 'score_before',
    'wickets_before', 'legal_balls_remaining_before', 'target', 'rrr_before', 'match_phase'
]].head(10)


## 5. Situational Pressure Analysis
We analyze how pressure manifests in the data:
- **RRR escalation**: In the 2nd innings, RRR variance explodes after over 12.
- **Death overs crunch**: Overs 16-20 demand maximum boundary hitting while preserving remaining wickets.
- **Dot ball pressure matrix**: As RRR increases above 10.0 and wickets drop below 4, dot ball percentage drops while risk of dismissal spikes.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# RRR by over in chase
chase_df = reconstructed_df[(reconstructed_df['innings'] == 2) & (reconstructed_df['rrr_before'].notnull())]
valid_rrr = chase_df[chase_df['rrr_before'] <= 24]

sns.boxplot(data=valid_rrr, x='overid', y='rrr_before', ax=axes[0], palette='magma_r', fliersize=1)
axes[0].axhline(10.0, color='red', linestyle='--', linewidth=1.5, label='High Pressure Cutoff (RRR >= 10.0)')
axes[0].set_title('Required Run Rate by Over (2nd Innings Chases)', fontsize=11, fontweight='bold')
axes[0].legend()

# Pressure heatmap
chase_sub = valid_rrr.copy()
chase_sub['rrr_bucket'] = pd.cut(chase_sub['rrr_before'], bins=[0, 6, 8, 10, 12, 15, 24], labels=['<6', '6-8', '8-10', '10-12', '12-15', '15+'])
chase_sub['wickets_bucket'] = pd.cut(chase_sub['wickets_in_hand_before'], bins=[-1, 2, 4, 6, 8, 10], labels=['0-2', '3-4', '5-6', '7-8', '9-10'])
chase_sub['is_dot'] = (chase_sub['runs'] == 0) & (chase_sub['extras1_runs'] == 0)
matrix = chase_sub.pivot_table(index='wickets_bucket', columns='rrr_bucket', values='is_dot', aggfunc='mean', observed=False) * 100

sns.heatmap(matrix.iloc[::-1], annot=True, fmt='.1f', cmap='coolwarm', cbar_kws={'label': 'Dot Ball %'}, ax=axes[1])
axes[1].set_title('Dot Ball % Landscape: Wickets in Hand vs RRR', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()


## 6. Next Steps for Phase 2: Pressure Quantification & Player Reliability
With the match situation reconstruction validated and verified:
1. Formalize pressure indicators for both Batters and Bowlers.
2. Formulate Clutch Differential Metrics:
   - Batter: $\Delta SR = SR_{pressure} - SR_{baseline}$, boundary conversion in crunch overs.
   - Bowler: $\Delta Econ = Econ_{baseline} - Econ_{pressure}$, dot ball rate and clutch wicket percentage.
3. Apply Bayesian Shrinkage / Regularization to eliminate low-sample noise and establish reproducible Top 10 rankings.
